# Phase 1 — Optimized ML Pipeline for AegisFin-AI

This notebook rebuilds Phase 1 using the original `application_train.csv` workflow, but with stronger leakage control, explicit imbalance handling, feature engineering, feature selection, Optuna tuning, and final comparison of:

- XGBoost
- LightGBM
- CatBoost

## Design goals

1. Understand and quantify class imbalance.
2. Keep useful information while handling missing values.
3. Avoid target leakage and preprocessing leakage.
4. Engineer financially meaningful ratio/age features.
5. Remove only clearly unusable features; do **not** blindly drop all columns with high missingness.
6. Use a train-only feature-selection stage.
7. Compare imbalance strategies, including `RandomOverSampler`, but prefer native class weighting when it is competitive.
8. Skip PCA unless an experiment demonstrates a real benefit.
9. Tune the three boosting models with Optuna using stratified cross-validation.
10. Evaluate using ROC-AUC, PR-AUC, Gini and KS rather than accuracy.
11. Refit the selected model on train + validation and evaluate the test set exactly once.


## Important corrections from the previous notebook

The previous notebook created several engineered ratio columns **after** creating `X`, so those new columns were not actually included in the trained feature matrix. The previous SHAP-based feature reduction was also performed using the validation set and then evaluated on that same validation set, which can make the reported performance optimistic.

This version fixes both issues by:

- creating row-level engineered features before the train/validation/test split;
- fitting feature selection only on the training partition;
- using the validation partition only for model/strategy selection;
- keeping the final test partition untouched until the final evaluation.


In [ ]:
# Colab / Jupyter package installation
# In VS Code, install the same packages once in your active environment instead.
!pip -q install -U xgboost lightgbm catboost optuna imbalanced-learn joblib


In [ ]:
import os
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 120)

print("Environment ready.")


In [ ]:
# Core ML / imbalance / optimization imports

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    classification_report,
)
from imblearn.over_sampling import RandomOverSampler

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from catboost import CatBoostClassifier

import optuna

print("optuna:", optuna.__version__)


## 1. Configuration

The notebook is intentionally configurable so you can first run a fast experiment and later increase the tuning budget.

For the first complete run, keep:

- `TUNE_ROWS = 80000`
- `N_TRIALS = 10`
- `CV_FOLDS = 3`

After the pipeline works correctly, increase the Optuna budget for the final experiment.


In [ ]:
# =========================
# USER CONFIGURATION
# =========================

DATA_CANDIDATES = [
    "/content/drive/MyDrive/Sample Data for colab/application_train.csv/application_train.csv",
    "/content/application_train.csv",
    "application_train.csv",
]

TEST_SIZE = 0.15
VALID_SIZE = 0.15

# Missingness policy
HIGH_MISSING_THRESHOLD = 0.80
MISSING_INDICATOR_THRESHOLD = 0.20

# Feature selection
TOP_K_FEATURES = 75
RUN_FEATURE_REDUCTION = True
REDUCTION_TOLERANCE = 0.995  # reduced model must retain at least 99.5% of validation PR-AUC

# Imbalance experiment
IMBALANCE_SAMPLE_ROWS = 80000
OVERSAMPLING_RATIO = 0.25  # minority becomes 25% of majority for the ROS experiment

# Optuna
TUNE_ROWS = 80000
CV_FOLDS = 3
N_TRIALS = 10
EARLY_STOPPING_ROUNDS = 60

# GPU toggle. Keep False for maximum portability.
USE_GPU = False

# Final artifact location
ARTIFACT_DIR = Path("./phase1_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

print("Configuration loaded.")


## 2. Load the dataset

The original notebook works with the Home Credit-style `application_train.csv` file and contains 307,511 rows and 122 original columns.

The path resolution below supports both Colab Drive and local/Jupyter execution.


In [ ]:
DATA_PATH = next((p for p in DATA_CANDIDATES if os.path.exists(p)), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "application_train.csv was not found. Update DATA_CANDIDATES in the configuration cell."
    )

data = pd.read_csv(DATA_PATH)

print("Loaded:", DATA_PATH)
print("Shape:", data.shape)
data.head()


In [ ]:
data.info()


In [ ]:
print("Target distribution:")
print(data["TARGET"].value_counts())

print("\nTarget percentage:")
print((data["TARGET"].value_counts(normalize=True) * 100).round(4))

imbalance_ratio = (
    data["TARGET"].value_counts()[0] /
    data["TARGET"].value_counts()[1]
)

print(f"\nMajority / minority ratio: {imbalance_ratio:.2f}:1")


In [ ]:
# Quick visual check of target imbalance

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(x=data["TARGET"], ax=ax)
ax.set_title("TARGET class distribution")
ax.set_xlabel("TARGET")
ax.set_ylabel("Count")
plt.show()


## 3. Basic data-quality audit

The dataset is highly imbalanced: roughly 92% of rows are `TARGET=0` and 8% are `TARGET=1` in the original file.

That means accuracy is not a useful primary metric. We will focus on:

- **PR-AUC / Average Precision** — especially important for an imbalanced positive class.
- **ROC-AUC** — ranking quality.
- **Gini** — `2 × ROC-AUC − 1`.
- **KS** — maximum separation between positive and negative score distributions.

We will still inspect duplicates, constant columns and missingness.


In [ ]:
missing_summary = (
    data.isna().mean()
    .sort_values(ascending=False)
    .rename("missing_fraction")
    .to_frame()
)

missing_summary["missing_percent"] = missing_summary["missing_fraction"] * 100
missing_summary.head(30)


In [ ]:
print("Duplicate rows:", data.duplicated().sum())

constant_columns = [
    c for c in data.columns
    if data[c].nunique(dropna=False) <= 1
]

print("Constant columns:", len(constant_columns))
print(constant_columns)


## 4. Feature engineering

These features are created **before** splitting, but they use only values from the same row and never use `TARGET`.

Important engineered features:

- credit-to-income ratio
- annuity-to-income ratio
- credit-to-annuity ratio
- goods-price-to-credit ratio
- income per family member
- income per child
- age in years
- employment duration
- external-source mean/std/min/max

The `DAYS_EMPLOYED = 365243` value is a sentinel-style value in this dataset. We convert it to missing and preserve a separate anomaly indicator rather than treating it as a real duration.


In [ ]:
def safe_divide(a, b):
    # Element-wise division that safely returns NaN for zero denominators.
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    out = pd.Series(np.nan, index=a.index, dtype="float64")
    valid = b.notna() & (b != 0) & a.notna()
    out.loc[valid] = a.loc[valid] / b.loc[valid]
    return out


def engineer_features(df):
    df = df.copy()

    # Employment anomaly
    if "DAYS_EMPLOYED" in df.columns:
        df["DAYS_EMPLOYED_ANOMALY"] = (
            df["DAYS_EMPLOYED"] == 365243
        ).astype("int8")
        df.loc[df["DAYS_EMPLOYED"] == 365243, "DAYS_EMPLOYED"] = np.nan

    # Age / employment
    if "DAYS_BIRTH" in df.columns:
        df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365.25

    if "DAYS_EMPLOYED" in df.columns:
        df["EMPLOYMENT_YEARS"] = -df["DAYS_EMPLOYED"] / 365.25

    # Financial ratios
    if {"AMT_CREDIT", "AMT_INCOME_TOTAL"}.issubset(df.columns):
        df["CREDIT_INCOME_RATIO"] = safe_divide(
            df["AMT_CREDIT"], df["AMT_INCOME_TOTAL"]
        )

    if {"AMT_ANNUITY", "AMT_INCOME_TOTAL"}.issubset(df.columns):
        df["ANNUITY_INCOME_RATIO"] = safe_divide(
            df["AMT_ANNUITY"], df["AMT_INCOME_TOTAL"]
        )

    if {"AMT_CREDIT", "AMT_ANNUITY"}.issubset(df.columns):
        df["CREDIT_ANNUITY_RATIO"] = safe_divide(
            df["AMT_CREDIT"], df["AMT_ANNUITY"]
        )

    if {"AMT_GOODS_PRICE", "AMT_CREDIT"}.issubset(df.columns):
        df["GOODS_CREDIT_RATIO"] = safe_divide(
            df["AMT_GOODS_PRICE"], df["AMT_CREDIT"]
        )

    if {"AMT_INCOME_TOTAL", "CNT_FAM_MEMBERS"}.issubset(df.columns):
        df["INCOME_PER_FAMILY_MEMBER"] = safe_divide(
            df["AMT_INCOME_TOTAL"],
            df["CNT_FAM_MEMBERS"] + 1
        )

    if {"AMT_INCOME_TOTAL", "CNT_CHILDREN"}.issubset(df.columns):
        df["INCOME_PER_CHILD"] = safe_divide(
            df["AMT_INCOME_TOTAL"],
            df["CNT_CHILDREN"] + 1
        )

    # External credit-score aggregates
    ext_cols = [
        c for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
        if c in df.columns
    ]

    if ext_cols:
        df["EXT_SOURCE_MEAN"] = df[ext_cols].mean(axis=1)
        df["EXT_SOURCE_STD"] = df[ext_cols].std(axis=1)
        df["EXT_SOURCE_MIN"] = df[ext_cols].min(axis=1)
        df["EXT_SOURCE_MAX"] = df[ext_cols].max(axis=1)

    return df


data_eng = engineer_features(data)

print("Original shape:", data.shape)
print("After engineering:", data_eng.shape)

engineered_cols = [
    c for c in data_eng.columns
    if c not in data.columns
]

print("Engineered columns:")
print(engineered_cols)


## 5. Train / validation / test split

We use three partitions:

- **Train (70%)** — preprocessing state, feature selection and Optuna CV.
- **Validation (15%)** — imbalance strategy and model comparison.
- **Test (15%)** — final one-time evaluation.

No oversampling is performed before this split.


In [ ]:
X_raw = data_eng.drop(columns=["TARGET"], errors="ignore")
y = data_eng["TARGET"].astype("int8")

X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X_raw,
    y,
    test_size=TEST_SIZE + VALID_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

relative_valid_size = VALID_SIZE / (TEST_SIZE + VALID_SIZE)

X_valid_raw, X_test_raw, y_valid, y_test = train_test_split(
    X_temp_raw,
    y_temp,
    test_size=1 - relative_valid_size,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

print("Train:", X_train_raw.shape, y_train.shape)
print("Valid:", X_valid_raw.shape, y_valid.shape)
print("Test :", X_test_raw.shape, y_test.shape)


## 6. Leakage-safe preprocessing state

We deliberately avoid aggressive imputation because XGBoost, LightGBM and CatBoost can work with missing numeric values.

Instead:

- remove the identifier `SK_ID_CURR`;
- drop only constant columns and columns above the 80% missing threshold;
- keep moderately/highly-missing features when they may contain signal;
- add missingness indicators for training columns with at least 20% missingness;
- encode categorical columns as pandas `category` with **training-derived category vocabularies**;
- do not scale numeric values because tree boosting does not require standardization.

### PCA decision

**PCA is intentionally skipped.**

The feature space is only around 120–130 engineered/original features, the selected models are tree-based, and PCA would reduce interpretability while potentially mixing useful financial/categorical signals into dense components.


In [ ]:
def fit_preprocessor(X_train):
    X = X_train.copy()

    drop_cols = set()

    # Identifier: useful for record lookup, not predictive learning.
    if "SK_ID_CURR" in X.columns:
        drop_cols.add("SK_ID_CURR")

    # Constant / all-null columns
    for col in X.columns:
        if X[col].nunique(dropna=False) <= 1:
            drop_cols.add(col)

    # Very high missingness: only the extreme case is dropped.
    missing_frac = X.isna().mean()
    high_missing_cols = missing_frac[
        missing_frac >= HIGH_MISSING_THRESHOLD
    ].index.tolist()

    drop_cols.update(high_missing_cols)

    X = X.drop(columns=sorted(drop_cols), errors="ignore")

    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    indicator_cols = [
        col for col in X.columns
        if X[col].isna().mean() >= MISSING_INDICATOR_THRESHOLD
    ]

    category_vocab = {}

    for col in categorical_cols:
        vals = X[col].astype("string").fillna("__MISSING__")
        categories = pd.Index(vals.unique().tolist(), dtype="string")

        if "__MISSING__" not in categories:
            categories = categories.append(
                pd.Index(["__MISSING__"], dtype="string")
            )

        category_vocab[col] = categories.tolist()

    state = {
        "drop_cols": sorted(drop_cols),
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "indicator_cols": indicator_cols,
        "category_vocab": category_vocab,
    }

    return state


def transform_features(X, state):
    X = X.copy()

    # Remove columns decided using training data only.
    X = X.drop(columns=state["drop_cols"], errors="ignore")

    # Add missingness indicators from the training-derived column list.
    for col in state["indicator_cols"]:
        if col in X.columns:
            X[f"{col}__MISSING"] = X[col].isna().astype("int8")

    # Numeric: keep NaN, because the selected tree models can handle it.
    for col in state["numeric_cols"]:
        if col in X.columns:
            X[col] = pd.to_numeric(X[col], errors="coerce")

    # Categorical: use the same category vocabulary across train/valid/test.
    for col in state["categorical_cols"]:
        if col in X.columns:
            vals = X[col].astype("string").fillna("__MISSING__")
            X[col] = pd.Categorical(
                vals,
                categories=state["category_vocab"][col]
            )

    return X


prep_state = fit_preprocessor(X_train_raw)

X_train = transform_features(X_train_raw, prep_state)
X_valid = transform_features(X_valid_raw, prep_state)
X_test = transform_features(X_test_raw, prep_state)

categorical_cols = prep_state["categorical_cols"]
numeric_cols = [
    c for c in X_train.columns
    if c not in categorical_cols
]

print("Dropped columns:", len(prep_state["drop_cols"]))
print("Categorical columns:", len(categorical_cols))
print("Numeric columns:", len(numeric_cols))
print("Final feature count before selection:", X_train.shape[1])
print("\nExample dropped columns:", prep_state["drop_cols"][:20])


In [ ]:
# Sanity checks after preprocessing
print("Train dtypes:")
print(X_train.dtypes.value_counts())

print("\nAny object columns remaining:", X_train.select_dtypes(include="object").columns.tolist())

print("\nTrain missingness summary:")
print(X_train.isna().mean().sort_values(ascending=False).head(15))


## 7. Imbalance handling experiment

The positive class is only about 8%.

We compare:

1. **No weighting**
2. **Mild weighting** = square root of the negative/positive ratio
3. **Full native weighting** = negative/positive ratio
4. **RandomOverSampler** to a 0.25 minority/majority ratio

For XGBoost and LightGBM, `scale_pos_weight` is designed for imbalanced binary classification; CatBoost provides `class_weights`.

We do not automatically use SMOTE here because the dataset contains mixed categorical and numerical features and is already large. Synthetic interpolation is not the first choice for this problem. The notebook keeps `RandomOverSampler` as a direct imbalanced-learn baseline instead.

The strategy is selected on the validation set using a fast LightGBM experiment, while the final model family and hyperparameters are still tuned separately.


In [ ]:
def calculate_pos_weight(y_values):
    counts = pd.Series(y_values).value_counts()
    return float(counts.get(0, 0) / max(counts.get(1, 1), 1))


def make_lgbm_quick(scale_pos_weight=1.0, seed=RANDOM_STATE):
    return LGBMClassifier(
        objective="binary",
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=50,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )


def evaluate_pr_auc(model, X_eval, y_eval):
    prob = model.predict_proba(X_eval)[:, 1]
    return average_precision_score(y_eval, prob), roc_auc_score(y_eval, prob)


# Fast, stratified training subset for strategy comparison.
imb_sample_n = min(IMBALANCE_SAMPLE_ROWS, len(X_train))

X_imb, _, y_imb, _ = train_test_split(
    X_train,
    y_train,
    train_size=imb_sample_n,
    stratify=y_train,
    random_state=RANDOM_STATE,
)

base_pos_weight = calculate_pos_weight(y_imb)

imbalance_candidates = {
    "no_weight": {"kind": "weight", "value": 1.0},
    "sqrt_weight": {"kind": "weight", "value": np.sqrt(base_pos_weight)},
    "full_weight": {"kind": "weight", "value": base_pos_weight},
    "random_over_sampler": {"kind": "ros", "value": OVERSAMPLING_RATIO},
}

imbalance_results = []

for name, cfg in imbalance_candidates.items():
    X_fit, y_fit = X_imb, y_imb

    if cfg["kind"] == "ros":
        ros = RandomOverSampler(
            sampling_strategy=cfg["value"],
            random_state=RANDOM_STATE,
        )
        X_fit, y_fit = ros.fit_resample(X_fit, y_fit)

    spw = cfg["value"] if cfg["kind"] == "weight" else 1.0

    model = make_lgbm_quick(scale_pos_weight=spw)
    model.fit(
        X_fit,
        y_fit,
        categorical_feature=[
            c for c in categorical_cols if c in X_fit.columns
        ],
    )

    pr_auc, roc_auc = evaluate_pr_auc(model, X_valid, y_valid)

    imbalance_results.append({
        "strategy": name,
        "scale_pos_weight": spw,
        "train_rows_after_resampling": len(X_fit),
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc,
    })

imbalance_results_df = (
    pd.DataFrame(imbalance_results)
    .sort_values("PR_AUC", ascending=False)
    .reset_index(drop=True)
)

display(imbalance_results_df)


In [ ]:
selected_imbalance_strategy = imbalance_results_df.iloc[0]["strategy"]
selected_imbalance_config = imbalance_candidates[selected_imbalance_strategy]

if selected_imbalance_config["kind"] == "weight":
    SELECTED_POS_WEIGHT = float(selected_imbalance_config["value"])
else:
    SELECTED_POS_WEIGHT = 1.0

print("Selected imbalance strategy:", selected_imbalance_strategy)
print("Selected native positive-class weight:", SELECTED_POS_WEIGHT)


## 8. Leakage-safe feature selection

Feature selection is performed using a LightGBM baseline fitted **only on the training partition**.

Why not simply remove every feature with low SHAP from the validation set?

Because selecting features from a validation set and then reporting performance on that same validation set can leak information into the evaluation.

We therefore:

- fit a baseline model on train only;
- rank features by training gain importance;
- keep the top `TOP_K_FEATURES`;
- always retain the engineered features so the ratio/aggregate information is not accidentally removed;
- compare all-features vs reduced-features on the validation partition.

The validation comparison is used only to decide whether the reduction is worth keeping; the test set remains untouched.


In [ ]:
# A modest baseline model used only for train-only feature ranking.
X_rank, _, y_rank, _ = train_test_split(
    X_train,
    y_train,
    train_size=min(150000, len(X_train)),
    stratify=y_train,
    random_state=RANDOM_STATE,
)

X_rank_balanced, y_rank_balanced = apply_training_imbalance_strategy(
    X_rank,
    y_rank,
)

rank_model = LGBMClassifier(
    objective="binary",
    n_estimators=350,
    learning_rate=0.04,
    num_leaves=31,
    min_child_samples=50,
    colsample_bytree=0.9,
    subsample=0.9,
    subsample_freq=1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=SELECTED_POS_WEIGHT,
    importance_type="gain",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)

rank_model.fit(
    X_rank_balanced,
    y_rank_balanced,
    categorical_feature=[
        c for c in categorical_cols if c in X_rank_balanced.columns
    ],
)

importance_df = (
    pd.DataFrame({
        "feature": X_rank.columns,
        "importance_gain": rank_model.feature_importances_,
    })
    .sort_values("importance_gain", ascending=False)
    .reset_index(drop=True)
)

display(importance_df.head(40))


In [ ]:
ENGINEERED_FEATURES = [
    c for c in [
        "DAYS_EMPLOYED_ANOMALY",
        "AGE_YEARS",
        "EMPLOYMENT_YEARS",
        "CREDIT_INCOME_RATIO",
        "ANNUITY_INCOME_RATIO",
        "CREDIT_ANNUITY_RATIO",
        "GOODS_CREDIT_RATIO",
        "INCOME_PER_FAMILY_MEMBER",
        "INCOME_PER_CHILD",
        "EXT_SOURCE_MEAN",
        "EXT_SOURCE_STD",
        "EXT_SOURCE_MIN",
        "EXT_SOURCE_MAX",
    ]
    if c in X_train.columns
]

top_ranked = importance_df["feature"].head(TOP_K_FEATURES).tolist()

selected_features = list(dict.fromkeys(
    ENGINEERED_FEATURES + top_ranked
))

selected_features = [
    c for c in selected_features
    if c in X_train.columns
]

print("Engineered features retained:", len(ENGINEERED_FEATURES))
print("Top-ranked features:", len(top_ranked))
print("Final selected feature count:", len(selected_features))

display(importance_df[importance_df["feature"].isin(selected_features)].head(50))


In [ ]:
X_train_reduced = X_train[selected_features].copy()
X_valid_reduced = X_valid[selected_features].copy()
X_test_reduced = X_test[selected_features].copy()

reduced_cat_cols = [
    c for c in categorical_cols if c in selected_features
]

# Compare a quick all-feature model against the reduced model
# using the exact same imbalance treatment selected earlier.
X_full_fit, y_full_fit = apply_training_imbalance_strategy(
    X_train.copy(),
    y_train.copy()
)

X_reduced_fit, y_reduced_fit = apply_training_imbalance_strategy(
    X_train_reduced.copy(),
    y_train.copy()
)

full_quick = make_lgbm_quick(
    scale_pos_weight=SELECTED_POS_WEIGHT
)
full_quick.fit(
    X_full_fit,
    y_full_fit,
    categorical_feature=[
        c for c in categorical_cols if c in X_full_fit.columns
    ],
)

reduced_quick = make_lgbm_quick(
    scale_pos_weight=SELECTED_POS_WEIGHT
)
reduced_quick.fit(
    X_reduced_fit,
    y_reduced_fit,
    categorical_feature=[
        c for c in reduced_cat_cols if c in X_reduced_fit.columns
    ],
)

full_prob = full_quick.predict_proba(X_valid)[:, 1]
reduced_prob = reduced_quick.predict_proba(X_valid_reduced)[:, 1]

full_pr_auc = average_precision_score(y_valid, full_prob)
reduced_pr_auc = average_precision_score(y_valid, reduced_prob)

full_roc_auc = roc_auc_score(y_valid, full_prob)
reduced_roc_auc = roc_auc_score(y_valid, reduced_prob)

print(f"All-feature PR-AUC    : {full_pr_auc:.6f}")
print(f"Reduced-feature PR-AUC: {reduced_pr_auc:.6f}")
print(f"All-feature ROC-AUC    : {full_roc_auc:.6f}")
print(f"Reduced-feature ROC-AUC: {reduced_roc_auc:.6f}")


In [ ]:
USE_REDUCED_FEATURES = (
    RUN_FEATURE_REDUCTION
    and reduced_pr_auc >= (full_pr_auc * REDUCTION_TOLERANCE)
)

if USE_REDUCED_FEATURES:
    X_train_model = X_train_reduced
    X_valid_model = X_valid_reduced
    X_test_model = X_test_reduced
    model_categorical_cols = reduced_cat_cols
else:
    X_train_model = X_train
    X_valid_model = X_valid
    X_test_model = X_test
    model_categorical_cols = [
        c for c in categorical_cols
        if c in X_train.columns
    ]

print("Using reduced features:", USE_REDUCED_FEATURES)
print("Features passed to final model family tuning:", X_train_model.shape[1])


## 9. Optuna tuning strategy

We use Optuna instead of a large Cartesian `GridSearchCV`.

Reason:

- the search space is continuous/discrete and fairly large;
- several parameters are strongly interacting;
- boosting models often need early stopping;
- Optuna can stop weak trials and explore promising regions more efficiently.

The objective is **mean PR-AUC across stratified folds**.

The tuning dataset is a stratified subset of the training partition only. The external validation and test partitions are not used by Optuna.

Official references used when designing this notebook:

- Optuna: https://optuna.readthedocs.io/
- XGBoost: https://xgboost.readthedocs.io/
- LightGBM: https://lightgbm.readthedocs.io/
- CatBoost: https://catboost.ai/docs/
- imbalanced-learn: https://imbalanced-learn.org/


In [ ]:
# Prepare a stratified subset for Optuna.
tune_n = min(TUNE_ROWS, len(X_train_model))

X_tune, _, y_tune, _ = train_test_split(
    X_train_model,
    y_train,
    train_size=tune_n,
    stratify=y_train,
    random_state=RANDOM_STATE,
)

print("Optuna tuning rows:", len(X_tune))
print("Positive rate:", y_tune.mean())


In [ ]:
def to_catboost_frame(X, categorical_columns):
    X_cb = X.copy()

    for col in categorical_columns:
        if col in X_cb.columns:
            X_cb[col] = (
                X_cb[col]
                .astype("string")
                .fillna("__MISSING__")
                .astype(str)
            )

    return X_cb


def make_xgb_model(params, use_early_stopping=True):
    xgb_kwargs = dict(
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        device="cuda" if USE_GPU else "cpu",
        enable_categorical=True,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    )

    if use_early_stopping:
        xgb_kwargs["early_stopping_rounds"] = EARLY_STOPPING_ROUNDS

    return XGBClassifier(
        **xgb_kwargs,
        **params,
    )


def make_lgbm_model(params):
    params = dict(params)
    params.pop("scale_pos_weight", None)

    return LGBMClassifier(
        objective="binary",
        metric="auc",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
        subsample_freq=1,
        scale_pos_weight=SELECTED_POS_WEIGHT,
        **params,
    )


def make_catboost_model(params):
    return CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=RANDOM_STATE,
        thread_count=-1,
        verbose=False,
        allow_writing_files=False,
        task_type="GPU" if USE_GPU else "CPU",
        class_weights=[1.0, SELECTED_POS_WEIGHT],
        **params,
    )


def apply_training_imbalance_strategy(X_fit, y_fit):
    # Native weighting is handled inside each model.
    # Only RandomOverSampler changes the actual training rows.
    if selected_imbalance_strategy == "random_over_sampler":
        ros = RandomOverSampler(
            sampling_strategy=OVERSAMPLING_RATIO,
            random_state=RANDOM_STATE,
        )
        return ros.fit_resample(X_fit, y_fit)

    return X_fit, y_fit


### 9.1 XGBoost Optuna objective


In [ ]:
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 400, 1200, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 25.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.65, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 20.0, log=True),
        "scale_pos_weight": SELECTED_POS_WEIGHT,
    }

    fold_scores = []
    cv = StratifiedKFold(
        n_splits=CV_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X_tune, y_tune)):
        X_tr = X_tune.iloc[tr_idx].copy()
        X_va = X_tune.iloc[va_idx].copy()
        y_tr = y_tune.iloc[tr_idx]
        y_va = y_tune.iloc[va_idx]

        X_tr, y_tr = apply_training_imbalance_strategy(X_tr, y_tr)

        model = make_xgb_model(params)
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False,
        )

        prob = model.predict_proba(X_va)[:, 1]
        score = average_precision_score(y_va, prob)
        fold_scores.append(score)

        partial = float(np.mean(fold_scores))
        trial.report(partial, step=fold)

        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_scores))


### 9.2 LightGBM Optuna objective


In [ ]:
def objective_lgbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 400, 1500, step=100),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
        "subsample": trial.suggest_float("subsample", 0.65, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 20.0, log=True),
    }

    fold_scores = []
    cv = StratifiedKFold(
        n_splits=CV_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X_tune, y_tune)):
        X_tr = X_tune.iloc[tr_idx].copy()
        X_va = X_tune.iloc[va_idx].copy()
        y_tr = y_tune.iloc[tr_idx]
        y_va = y_tune.iloc[va_idx]

        X_tr, y_tr = apply_training_imbalance_strategy(X_tr, y_tr)

        model = make_lgbm_model(params)
        model.fit(
            X_tr,
            y_tr,
            categorical_feature=[
                c for c in model_categorical_cols
                if c in X_tr.columns
            ],
            eval_set=[(X_va, y_va)],
            eval_metric="auc",
            callbacks=[
                early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
                log_evaluation(0),
            ],
        )

        prob = model.predict_proba(X_va)[:, 1]
        score = average_precision_score(y_va, prob)
        fold_scores.append(score)

        partial = float(np.mean(fold_scores))
        trial.report(partial, step=fold)

        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_scores))


### 9.3 CatBoost Optuna objective


In [ ]:
def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 500, 1600, step=100),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.0, 3.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),
        "border_count": trial.suggest_int("border_count", 64, 254),
    }

    fold_scores = []
    cv = StratifiedKFold(
        n_splits=CV_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X_tune, y_tune)):
        X_tr = X_tune.iloc[tr_idx].copy()
        X_va = X_tune.iloc[va_idx].copy()
        y_tr = y_tune.iloc[tr_idx]
        y_va = y_tune.iloc[va_idx]

        X_tr, y_tr = apply_training_imbalance_strategy(X_tr, y_tr)

        X_tr_cb = to_catboost_frame(X_tr, model_categorical_cols)
        X_va_cb = to_catboost_frame(X_va, model_categorical_cols)

        model = make_catboost_model(params)

        model.fit(
            X_tr_cb,
            y_tr,
            cat_features=[
                c for c in model_categorical_cols
                if c in X_tr_cb.columns
            ],
            eval_set=(X_va_cb, y_va),
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose=False,
        )

        prob = model.predict_proba(X_va_cb)[:, 1]
        score = average_precision_score(y_va, prob)
        fold_scores.append(score)

        partial = float(np.mean(fold_scores))
        trial.report(partial, step=fold)

        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_scores))


## 10. Run the Optuna studies

The studies optimize **PR-AUC**, not accuracy.

Run all three so we have a fair, documented comparison. Do not compare models on different preprocessing or different imbalance logic.


In [ ]:
def run_optuna_study(objective_fn, study_name, n_trials=N_TRIALS):
    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=max(3, min(5, n_trials // 2)),
        n_warmup_steps=1,
    )

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        study_name=study_name,
    )

    study.optimize(
        objective_fn,
        n_trials=n_trials,
        show_progress_bar=True,
    )

    print(f"{study_name} best PR-AUC:", study.best_value)
    print("Best parameters:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")

    return study


In [ ]:
# Run one model at a time so notebook output remains manageable.

xgb_study = run_optuna_study(
    objective_xgb,
    study_name="phase1_xgboost_pr_auc",
)

lgbm_study = run_optuna_study(
    objective_lgbm,
    study_name="phase1_lightgbm_pr_auc",
)

cat_study = run_optuna_study(
    objective_catboost,
    study_name="phase1_catboost_pr_auc",
)


## 11. Train tuned models on the full training partition

The tuned parameters are now fixed. We train each model on the complete training partition and evaluate on validation.

The test partition is still untouched.


In [ ]:
best_xgb_params = dict(xgb_study.best_params)
best_lgbm_params = dict(lgbm_study.best_params)
best_cat_params = dict(cat_study.best_params)

# Keep the selected imbalance treatment fixed.
best_xgb_params["scale_pos_weight"] = SELECTED_POS_WEIGHT

X_train_fit = X_train_model.copy()
y_train_fit = y_train.copy()

X_valid_fit = X_valid_model.copy()
y_valid_fit = y_valid.copy()

X_test_fit = X_test_model.copy()
y_test_fit = y_test.copy()

# Optional actual row resampling for the oversampling strategy.
X_train_fit_balanced, y_train_fit_balanced = apply_training_imbalance_strategy(
    X_train_fit, y_train_fit
)

print("Original training rows:", len(X_train_fit))
print("Rows used by training:", len(X_train_fit_balanced))


In [ ]:
xgb_final = make_xgb_model(best_xgb_params)
xgb_final.fit(
    X_train_fit_balanced,
    y_train_fit_balanced,
    eval_set=[(X_valid_fit, y_valid_fit)],
    verbose=False,
)

lgbm_final = make_lgbm_model(best_lgbm_params)
lgbm_final.fit(
    X_train_fit_balanced,
    y_train_fit_balanced,
    categorical_feature=[
        c for c in model_categorical_cols
        if c in X_train_fit_balanced.columns
    ],
    eval_set=[(X_valid_fit, y_valid_fit)],
    eval_metric="auc",
    callbacks=[
        early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
        log_evaluation(0),
    ],
)

X_train_cb = to_catboost_frame(
    X_train_fit_balanced,
    model_categorical_cols
)
X_valid_cb = to_catboost_frame(
    X_valid_fit,
    model_categorical_cols
)

cat_final = make_catboost_model(best_cat_params)
cat_final.fit(
    X_train_cb,
    y_train_fit_balanced,
    cat_features=[
        c for c in model_categorical_cols
        if c in X_train_cb.columns
    ],
    eval_set=(X_valid_cb, y_valid_fit),
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    verbose=False,
)

print("All three tuned models trained.")


## 12. Unified evaluation

We report four useful ranking metrics.

For the positive-class imbalance, **PR-AUC is especially important** because it focuses on precision/recall quality for the minority class.

We also calculate:

- ROC-AUC
- Gini
- KS


In [ ]:
def ks_statistic(y_true, y_prob):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return float(np.max(tpr - fpr))


def evaluate_model(name, model, X_eval, y_eval, catboost=False):
    X_use = (
        to_catboost_frame(X_eval, model_categorical_cols)
        if catboost else X_eval
    )

    prob = model.predict_proba(X_use)[:, 1]

    roc_auc = roc_auc_score(y_eval, prob)
    pr_auc = average_precision_score(y_eval, prob)
    gini = 2 * roc_auc - 1
    ks = ks_statistic(y_eval, prob)

    return {
        "Model": name,
        "ROC_AUC": roc_auc,
        "PR_AUC": pr_auc,
        "Gini": gini,
        "KS": ks,
    }, prob


model_results = []
validation_probabilities = {}

for name, model, X_eval, is_cb in [
    ("XGBoost", xgb_final, X_valid_fit, False),
    ("LightGBM", lgbm_final, X_valid_fit, False),
    ("CatBoost", cat_final, X_valid_fit, True),
]:
    result, prob = evaluate_model(
        name,
        model,
        X_eval,
        y_valid_fit,
        catboost=is_cb,
    )
    model_results.append(result)
    validation_probabilities[name] = prob

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values("PR_AUC", ascending=False)
    .reset_index(drop=True)
)

display(model_results_df)


## 13. Select the final model for Phase 1

This is **model selection**, not a claim that one algorithm is universally best.

The notebook selects the candidate with the highest validation PR-AUC, then checks the same candidate once on the untouched test set.

A future production/business phase can add probability calibration and a business-specific approval/rejection threshold.


In [ ]:
best_model_name = model_results_df.iloc[0]["Model"]

model_lookup = {
    "XGBoost": xgb_final,
    "LightGBM": lgbm_final,
    "CatBoost": cat_final,
}

best_model = model_lookup[best_model_name]

print("Selected validation champion for Phase 1:", best_model_name)
print(model_results_df)


## 14. Threshold analysis on validation

A default probability threshold of 0.50 is rarely ideal for an imbalanced classification problem.

We therefore compute the threshold that maximizes F1 on the validation set and also report the KS-optimal threshold. These thresholds are **analysis outputs**, not hardcoded business rules.


In [ ]:
best_prob = validation_probabilities[best_model_name]

precision, recall, thresholds = precision_recall_curve(y_valid_fit, best_prob)

f1_scores = (
    2 * precision[:-1] * recall[:-1] /
    np.maximum(precision[:-1] + recall[:-1], 1e-12)
)

best_f1_idx = int(np.argmax(f1_scores))
best_f1_threshold = float(thresholds[best_f1_idx])

fpr, tpr, ks_thresholds = roc_curve(y_valid_fit, best_prob)
ks_values = tpr - fpr
best_ks_idx = int(np.argmax(ks_values))
best_ks_threshold = float(ks_thresholds[best_ks_idx])

print(f"Validation F1-optimal threshold: {best_f1_threshold:.6f}")
print(f"Validation KS-optimal threshold: {best_ks_threshold:.6f}")


## 15. Final refit on train + validation

The validation scores above are used to select the model family and hyperparameters. Now we refit the selected model on **train + validation** so the final model can use all development data before the untouched test evaluation.

We keep the original train-fitted preprocessing state so the feature schema remains identical to what was tuned and what the downstream application will reproduce.


In [ ]:
# Combine development data while preserving the tuned feature schema.
X_dev = pd.concat(
    [X_train_fit, X_valid_fit],
    axis=0,
    ignore_index=True,
)
y_dev = pd.concat(
    [y_train_fit.reset_index(drop=True), y_valid_fit.reset_index(drop=True)],
    axis=0,
    ignore_index=True,
)

X_dev_balanced, y_dev_balanced = apply_training_imbalance_strategy(
    X_dev,
    y_dev,
)

def get_best_iteration(model_name, model, fallback_params):
    if model_name == "XGBoost":
        best_it = getattr(model, "best_iteration", None)
        if best_it is not None:
            return int(best_it) + 1
        return int(fallback_params["n_estimators"])

    if model_name == "LightGBM":
        best_it = getattr(model, "best_iteration_", None)
        if best_it is not None:
            return int(best_it)
        return int(fallback_params["n_estimators"])

    if model_name == "CatBoost":
        try:
            best_it = model.get_best_iteration()
            if best_it is not None and best_it >= 0:
                return int(best_it) + 1
        except Exception:
            pass
        return int(fallback_params["iterations"])

    raise ValueError("Unknown model name")


final_n_estimators = get_best_iteration(
    "XGBoost", xgb_final, best_xgb_params
)
final_lgbm_estimators = get_best_iteration(
    "LightGBM", lgbm_final, best_lgbm_params
)
final_cat_iterations = get_best_iteration(
    "CatBoost", cat_final, best_cat_params
)

final_xgb_params = dict(best_xgb_params)
final_lgbm_params = dict(best_lgbm_params)
final_cat_params = dict(best_cat_params)

final_xgb_params["n_estimators"] = final_n_estimators
final_lgbm_params["n_estimators"] = final_lgbm_estimators
final_cat_params["iterations"] = final_cat_iterations

# Refit all three models on train + validation.
xgb_final_refit = make_xgb_model(
    final_xgb_params,
    use_early_stopping=False,
)
xgb_final_refit.fit(
    X_dev_balanced,
    y_dev_balanced,
    verbose=False,
)

lgbm_final_refit = make_lgbm_model(final_lgbm_params)
lgbm_final_refit.fit(
    X_dev_balanced,
    y_dev_balanced,
    categorical_feature=[
        c for c in model_categorical_cols
        if c in X_dev_balanced.columns
    ],
)

X_dev_cb = to_catboost_frame(
    X_dev_balanced,
    model_categorical_cols
)

cat_final_refit = make_catboost_model(final_cat_params)
cat_final_refit.fit(
    X_dev_cb,
    y_dev_balanced,
    cat_features=[
        c for c in model_categorical_cols
        if c in X_dev_cb.columns
    ],
    verbose=False,
)

final_model_lookup = {
    "XGBoost": xgb_final_refit,
    "LightGBM": lgbm_final_refit,
    "CatBoost": cat_final_refit,
}

best_model = final_model_lookup[best_model_name]

print("Development rows used for final refit:", len(X_dev))
print("Final XGBoost estimators:", final_n_estimators)
print("Final LightGBM estimators:", final_lgbm_estimators)
print("Final CatBoost iterations:", final_cat_iterations)


## 15. Final test evaluation — touched once

The test set is now evaluated using the selected model.

Do not iterate on the model after looking at this test result. Any subsequent changes should go back to train/validation and the test should be replaced by a fresh holdout.


In [ ]:
# Evaluate the final refitted model on the untouched test set.
if best_model_name == "CatBoost":
    X_test_eval = to_catboost_frame(
        X_test_fit,
        model_categorical_cols
    )
else:
    X_test_eval = X_test_fit

test_prob = best_model.predict_proba(X_test_eval)[:, 1]

test_roc_auc = roc_auc_score(y_test_fit, test_prob)
test_pr_auc = average_precision_score(y_test_fit, test_prob)
test_gini = 2 * test_roc_auc - 1
test_ks = ks_statistic(y_test_fit, test_prob)

final_test_results = pd.DataFrame([{
    "Model": best_model_name,
    "Test_ROC_AUC": test_roc_auc,
    "Test_PR_AUC": test_pr_auc,
    "Test_Gini": test_gini,
    "Test_KS": test_ks,
}])

display(final_test_results)


## 16. Final feature importance

Feature importance is extracted from the selected model.

Remember: tree-model importance is an interpretation aid, not proof of causality. A later explainability phase can add SHAP on a controlled sample.


In [ ]:
def get_feature_importance(model_name, model, feature_names):
    if model_name == "LightGBM":
        imp = model.feature_importances_
    elif model_name == "XGBoost":
        imp = model.feature_importances_
    elif model_name == "CatBoost":
        imp = model.get_feature_importance()
    else:
        raise ValueError("Unknown model")

    return (
        pd.DataFrame({
            "feature": feature_names,
            "importance": imp,
        })
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )


final_importance = get_feature_importance(
    best_model_name,
    best_model,
    X_train_model.columns,
)

display(final_importance.head(40))


In [ ]:
plt.figure(figsize=(10, 10))

top_imp = final_importance.head(25).sort_values("importance")

plt.barh(top_imp["feature"], top_imp["importance"])
plt.title(f"Top 25 Feature Importances — {best_model_name}")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 17. Optional SHAP analysis

Run this only after the main model is stable. SHAP can be expensive on large datasets, so we explain a sample rather than the entire dataset.

This section is interpretation only and is **not** used to choose features or score the test set.


In [ ]:
RUN_SHAP = False

if RUN_SHAP:
    import shap

    shap_n = min(3000, len(X_valid_fit))
    X_shap = X_valid_fit.sample(shap_n, random_state=RANDOM_STATE)

    if best_model_name == "CatBoost":
        X_shap_input = to_catboost_frame(
            X_shap,
            model_categorical_cols
        )
    else:
        X_shap_input = X_shap

    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_shap_input)

    if isinstance(shap_values, list):
        shap_values_plot = shap_values[-1]
    else:
        shap_values_plot = shap_values

    shap.summary_plot(
        shap_values_plot,
        X_shap_input,
        max_display=25,
    )
else:
    print("Set RUN_SHAP = True to run optional SHAP analysis.")


## 18. Save artifacts for later integration

The goal is to make Phase 1 directly reusable by the later AegisFin-AI application.

We save:

- final feature list;
- preprocessing state;
- imbalance configuration;
- validation/test metrics;
- Optuna best parameters;
- model in its native format.

The preprocessing state is essential: the downstream application must apply the **same** feature engineering, dropped columns, missing indicators and categorical vocabularies.


In [ ]:
# Save metadata
metadata = {
    "model_name": best_model_name,
    "selected_imbalance_strategy": selected_imbalance_strategy,
    "selected_pos_weight": float(SELECTED_POS_WEIGHT),
    "use_reduced_features": bool(USE_REDUCED_FEATURES),
    "feature_count": int(X_train_model.shape[1]),
    "selected_features": list(X_train_model.columns),
    "categorical_features": list(model_categorical_cols),
    "validation_results": model_results_df.to_dict(orient="records"),
    "test_results": final_test_results.to_dict(orient="records"),
    "best_params": {
        "XGBoost": final_xgb_params,
        "LightGBM": final_lgbm_params,
        "CatBoost": final_cat_params,
    },
    "validation_thresholds": {
        "f1_optimal": float(best_f1_threshold),
        "ks_optimal": float(best_ks_threshold),
    },
}

with open(ARTIFACT_DIR / "phase1_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

joblib.dump(
    prep_state,
    ARTIFACT_DIR / "preprocessing_state.joblib"
)

joblib.dump(
    pd.Series(X_train_model.columns.tolist()),
    ARTIFACT_DIR / "selected_features.joblib"
)

# Save final selected model using its native format.
if best_model_name == "XGBoost":
    best_model.save_model(str(ARTIFACT_DIR / "best_xgboost.json"))
elif best_model_name == "LightGBM":
    best_model.booster_.save_model(
        str(ARTIFACT_DIR / "best_lightgbm.txt")
    )
elif best_model_name == "CatBoost":
    best_model.save_model(
        str(ARTIFACT_DIR / "best_catboost.cbm")
    )

print("Artifacts saved to:", ARTIFACT_DIR.resolve())


# Final Phase 1 checklist

### Data
- [x] Target imbalance measured.
- [x] Duplicate rows checked.
- [x] Constant/high-missing features audited.
- [x] Numeric missing values preserved for tree models.
- [x] Missingness indicators added where useful.
- [x] Identifier removed.

### Leakage control
- [x] Train/validation/test split done before learned preprocessing.
- [x] Category vocabulary learned from train only.
- [x] Feature selection learned from train only.
- [x] Optuna uses train-only CV.
- [x] Test set used once at the end.

### Feature engineering
- [x] Credit/income ratios.
- [x] Annuity/income ratio.
- [x] Credit/annuity ratio.
- [x] Goods/credit ratio.
- [x] Age and employment duration.
- [x] External-source aggregates.
- [x] Employment sentinel handling.

### Imbalance
- [x] Native weighting tested.
- [x] RandomOverSampler tested.
- [x] Validation PR-AUC used for comparison.
- [x] No oversampling before the train/test split.

### Models
- [x] XGBoost.
- [x] LightGBM.
- [x] CatBoost.
- [x] Optuna hyperparameter tuning.
- [x] Early stopping.

### Dimensionality reduction
- [x] PCA skipped intentionally because the final algorithms are tree boosters and the feature count is moderate.

### Evaluation
- [x] PR-AUC.
- [x] ROC-AUC.
- [x] Gini.
- [x] KS.
- [x] Validation threshold analysis.
- [x] One-time final test evaluation.


## Next phase boundary

The output of this notebook is the **trained Phase 1 risk-classification model + preprocessing state**.

For integration into AegisFin-AI, the next step is to wrap the saved preprocessing state and model into a clean inference function, for example:

`raw application data -> feature engineering -> same preprocessing -> selected features -> model.predict_proba()`

Probability calibration and business thresholding should be handled as separate steps rather than mixed into the core model-training notebook.
